### Mini Project 2 \(By Juan Carlos Ferreyra\)

Will This Be a Strong Sales Week?
Using data from Walmart, we will be using probability, Bayes Theorem and Naive Bayes to identify sales patterns


### Event Definitions

S: Strong Week \(sales in a week are 10% above store's average\)

H: Holiday Week \(Where the holiday\_flag variable is present\)

Q: Holiday Season \(November or December\)

C: Cold Week \(Temperature reaches below 40 degrees Fahrenheit

F: Expensive Fules \(Fuel price is $3\.50 or more\)

J: High Unemployment \(At least 8%\)

G: Large Store \(Weekly sales reaching $1 Million\+\)

In [1]:
import numpy as np
import pandas as pd
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import confusion_matrix,accuracy_score,precision_score,recall_score

### Part P

We upload the data to conduct the analytics \(P1\)

In [2]:
wm=pd.read_csv("Walmart.csv")
wm["Date"]=pd.to_datetime(wm["Date"],dayfirst=True)
print(len(wm))
print(wm["Store"].nunique())
print(wm["Date"].nunique())
print(wm["Holiday_Flag"].sum())

6435
45
143
450


We have 6435 rows, with 45 stores, 143 weeks, and 450 holiday store\-weeks

We create columns with the probability of each event \(P2\)

In [3]:
avg = wm.groupby("Store")["Weekly_Sales"].mean()
ev = wm.copy()
avg= ev["Store"].map(avg)
ev["S"] = ev["Weekly_Sales"] >= 1.10 * avg
ev["H"] = ev["Holiday_Flag"] == 1
ev["Q"] = ev["Date"].dt.month.isin([11, 12])
ev["C"] = ev["Temperature"] < 40
ev["F"] = ev["Fuel_Price"] >= 3.50
ev["J"] = ev["Unemployment"] >= 8
ev["G"] = avg >= 1_000_000
ev.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,S,H,Q,C,F,J,G
0,1,2010-02-05,1643690.90,0,42.31,2.572,211.096358,8.106,False,False,False,False,False,True,True
1,1,2010-02-12,1641957.44,1,38.51,2.548,211.242170,8.106,False,True,False,True,False,True,True
2,1,2010-02-19,1611968.17,0,39.93,2.514,211.289143,8.106,False,False,False,True,False,True,True
3,1,2010-02-26,1409727.59,0,46.63,2.561,211.319643,8.106,False,False,False,False,False,True,True
4,1,2010-03-05,1554806.68,0,46.50,2.625,211.350143,8.106,False,False,False,False,False,True,True


In [4]:
events = ["S", "H", "Q", "C", "F", "J", "G"]
results = []
for e in events:
    count = ev[e].sum()
    probability = count / len(ev)
    complement = (~ev[e]).sum() / len(ev)
    results.append({"Event": e,"Count": count,"Probability": round(probability, 4),"Complement": round(complement, 4),"Check": round(probability + complement, 4)})
prob_table = pd.DataFrame(results)
display(prob_table)

,Event,Count,Probability,Complement,Check
0,S,814,0.1265,0.8735,1.0
1,H,450,0.0699,0.9301,1.0
2,Q,810,0.1259,0.8741,1.0
3,C,989,0.1537,0.8463,1.0
4,F,3029,0.4707,0.5293,1.0
5,J,3022,0.4696,0.5304,1.0
6,G,3146,0.4889,0.5111,1.0


By dividing the number of store\_weeks in the event by the total number of rows, we can calculate the probability of each event, also varifying using R5 the complement

We then generate the joint probabilities \(P3\), which includes Strong week and Holiday week, Strong week and November or December, Holiday week and November or December, and Strong week and Cold week

In [5]:
p_sh = (ev["S"] & ev["H"]).mean()
p_sq = (ev["S"] & ev["Q"]).mean()
p_hq = (ev["H"] & ev["Q"]).mean()
p_sc = (ev["S"] & ev["C"]).mean()
print("P(S ∩ H):", round(p_sh, 4))
print("P(S ∩ Q):", round(p_sq, 4))
print("P(H ∩ Q):", round(p_hq, 4))
print("P(S ∩ C):", round(p_sc, 4))

P(S ∩ H): 0.019
P(S ∩ Q): 0.0517
P(H ∩ Q): 0.028
P(S ∩ C): 0.0287


By counting the store\-weeks where both events occured by the total number of observations, we can find the joint probability of the described events

### Part B

We find the conditional probability regarding holidays \(B1\)

In [6]:
p_s_h = ev.loc[ev["H"], "S"].mean()
p_s_not_h = ev.loc[~ev["H"], "S"].mean()
print("P(S | H):", round(p_s_h, 4))
print("P(S | Hc):", round(p_s_not_h, 4))

P(S | H): 0.2711
P(S | Hc): 0.1156


Considering the probabilities, a strong sales week is more likely when it is a holiday week \(around 27%\) compared to a non\-holiday week \(around 12%\)

We now calculate the same through Bayes theorem \(B2\)

In [7]:
p_h = ev["H"].mean()
p_s = ev["S"].mean()
p_h_s = (p_s_h * p_h) / p_s
print("P(H | S) using Bayes:", round(p_h_s, 4))
print("P(H | S) by counting:", round(ev.loc[ev["S"], "H"].mean(), 4))

P(H | S) using Bayes: 0.1499
P(H | S) by counting: 0.1499


We see that both values match, giving around 15%, which means that around 15% of strong sles weeks occur during a holiday week

We now calculate the total probability over the holidays \(B3\)

In [8]:
ev["Holiday"] = "No Holiday"
ev.loc[ev["Date"].isin(pd.to_datetime(["2010-02-12", "2011-02-11", "2012-02-10"])), "Holiday"] = "Super Bowl"
ev.loc[ev["Date"].isin(pd.to_datetime(["2010-09-10", "2011-09-09", "2012-09-07"])), "Holiday"] = "Labor Day"
ev.loc[ev["Date"].isin(pd.to_datetime(["2010-11-26", "2011-11-25"])), "Holiday"] = "Thanksgiving"
ev.loc[ev["Date"].isin(pd.to_datetime(["2010-12-31", "2011-12-30"])), "Holiday"] = "Christmas"
b3 = ev.groupby("Holiday")["S"].agg(["count", "sum", "mean"])
b3["P(g)"] = b3["count"] / len(ev)
b3["P(S|g)"] = b3["mean"]
b3["P(g|S)"] = b3["sum"] / ev["S"].sum()
print(b3[["P(g)", "P(S|g)", "P(g|S)"]])
print("Total probability:",(b3["P(g)"] * b3["P(S|g)"]).sum())
print("P(S):", ev["S"].mean())
print("Highest holiday:", b3.drop("No Holiday")["P(S|g)"].idxmax())
print("Largest share of strong weeks:", b3["P(g|S)"].idxmax())

                  P(g)    P(S|g)    P(g|S)
Holiday                                   
Christmas     0.013986  0.066667  0.007371
Labor Day     0.020979  0.118519  0.019656
No Holiday    0.930070  0.115622  0.850123
Super Bowl    0.020979  0.192593  0.031941
Thanksgiving  0.013986  0.822222  0.090909
Total probability: 0.12649572649572652
P(S): 0.1264957264957265
Highest holiday: Thanksgiving
Largest share of strong weeks: No Holiday


The holiday with the highest probability of a strong sales week is Thanksgiving with around 82%, with around 85% of all strong weeks occuring outside of a holiday\. We then use the law of total probability and the average calculated through the dataset and confirm both values match\.

Now we calculate conditional probability through season \(B4\)

In [9]:
p_s_q = ev.loc[ev["Q"], "S"].mean()
p_q_s = ev.loc[ev["S"], "Q"].mean()
print("P(S | Q):", round(p_s_q, 4))
print("P(Q | S):", round(p_q_s, 4))

P(S | Q): 0.4111
P(Q | S): 0.4091


On one hand, P\(S\|Q\) measures the proportion of November or Decemebr weeks that are strong, while at the other hand, P\(Q\|S\) measures the proportion of strong weeks that occured in November or December

Now, we evaluate the staffing rule with Bayes Theorem \(B5\)

In [10]:
holiday_precision = (p_h_s * p_s) / (p_h_s * p_s + ev.loc[~ev["S"], "H"].mean() * (1 - p_s))
holiday_recall = ev.loc[ev["S"], "H"].mean()
season_precision = ev.loc[ev["Q"], "S"].mean()
season_recall = ev.loc[ev["S"], "Q"].mean()
print("Holiday precision:", round(holiday_precision, 4))
print("Holiday recall:", round(holiday_recall, 4))
print("Season precision:", round(season_precision, 4))
print("Season recall:", round(season_recall, 4))

Holiday precision: 0.2711
Holiday recall: 0.1499
Season precision: 0.4111
Season recall: 0.4091


The holiday rule has a precision of around 27% and a recall of around 15%, on the other hand, the November or December rule has a precision of around 41% and recall of around 41% as well\. On both measures, the November or December rule performs better\.

Now, we will be calculating the base rate \(B6\)

In [11]:
p_h_not_s = ev.loc[~ev["S"], "H"].mean()
p_double = 2 * p_s
precision_double = (p_h_s * p_double) / (p_h_s * p_double + p_h_not_s * (1 - p_double))
p_half = p_s / 2
precision_half = (p_h_s * p_half) / (p_h_s * p_half + p_h_not_s * (1 - p_half))
print("Original precision:", round(holiday_precision, 4))
print("Double strong weeks:", round(precision_double, 4))
print("Half strong weeks:", round(precision_half, 4))

Original precision: 0.2711
Double strong weeks: 0.4652
Half strong weeks: 0.1478


Given the strong weeks increasing in frequency twice as much, the holiday's precision rule would increase to around 47%, with having the frequency decrease by have, the holiday's precision rule would decrease to around 15%

We now conduct independence checks \(B7\)

In [12]:
p_s = ev["S"].mean()
print("H:", (ev["S"] & ev["H"]).mean(), p_s * ev["H"].mean())
print("Q:", (ev["S"] & ev["Q"]).mean(), p_s * ev["Q"].mean())
print("C:", (ev["S"] & ev["C"]).mean(), p_s * ev["C"].mean())
print("F:", (ev["S"] & ev["F"]).mean(), p_s * ev["F"].mean())
print("J:", (ev["S"] & ev["J"]).mean(), p_s * ev["J"].mean())
print("G:", (ev["S"] & ev["G"]).mean(), p_s * ev["G"].mean())

H: 0.01895881895881896 0.008845854999701154
Q: 0.05174825174825175 0.015922538999462076
C: 0.028749028749028748 0.019441223543787644
F: 0.04522144522144522 0.0595424328757662
J: 0.06402486402486403 0.05940483068688197
G: 0.05703185703185703 0.06184235517568851


None of the joint probabilities are equal to the products of their individual probabilities, meaning that they are all dependent on S\. The largest differences we can observed are Q\(November or December\), F \(Expensive Fuel\), and H\(Holidays\), which based on this comparisson seem to provide the most information about strong sales weeks\.

### Part N

We separate the data into training and testing data \(N1\)

In [13]:
def add_events(d, store_avg):
    d = d.copy()
    avg = d["Store"].map(store_avg)
    d["S"] = d["Weekly_Sales"] >= 1.10 * avg
    d["H"] = d["Holiday_Flag"] == 1
    d["Q"] = d["Date"].dt.month.isin([11, 12])
    d["C"] = d["Temperature"] < 40
    d["F"] = d["Fuel_Price"] >= 3.50
    d["J"] = d["Unemployment"] >= 8
    d["G"] = avg >= 1_000_000
    return d
train_raw = wm[wm["Date"].dt.year <= 2011]
test_raw = wm[wm["Date"].dt.year == 2012]
store_avg = train_raw.groupby("Store")["Weekly_Sales"].mean()
train = add_events(train_raw, store_avg)
test = add_events(test_raw, store_avg)
print("Training P(S):", round(train["S"].mean(), 4))
print("Testing P(S):", round(test["S"].mean(), 4))
print("Training check:", train["S"].sum() / len(train))
print("Testing check:", test["S"].sum() / len(test))

Training P(S): 0.1276
Testing P(S): 0.1318
Training check: 0.12755555555555556
Testing check: 0.13178294573643412


The probability of a strong week was around 13% in the training set and in the testing set \(12\.76% and 13\.18% precisely\)\. The frequency of strong weeks remained consistent between these two periods of time

We are now calculating prior probabilities and calculating using only training data \(N2\)

In [18]:
p_s = train["S"].mean()
p_not_s = 1 - p_s
print("P(S):", round(p_s, 4))
print("P(Sc):", round(p_not_s, 4))
features = ["H", "Q", "C", "F", "J", "G"]
p_feature_s = train[train["S"]][features].mean()
p_feature_not_s = train[~train["S"]][features].mean()
print(pd.DataFrame({
    "P(feature|S)": p_feature_s,
    "P(feature|Sc)": p_feature_not_s
}))

P(S): 0.1276
P(Sc): 0.8724
   P(feature|S)  P(feature|Sc)
H      0.172474       0.066480
Q      0.583624       0.120988
C      0.287456       0.152573
F      0.188153       0.345899
J      0.552265       0.531075
G      0.463415       0.492613


Around 13% of teh training data was made up by strong weeks, with non\-strong weeks made up around 87%\. Among strong weeks, November or December, Holidays and cold weather were the most common\. 

Now, we will be doing Naive Bayes predictions \(N3\)

In [22]:
a_s = p_s * p_feature_s["H"] * p_feature_s["Q"] * p_feature_s["C"] * (1-p_feature_s["F"]) * (1-p_feature_s["J"]) * p_feature_s["G"]
a_not_s = p_not_s * p_feature_not_s["H"] * p_feature_not_s["Q"] * p_feature_not_s["C"] * (1-p_feature_not_s["F"]) * (1-p_feature_not_s["J"]) * p_feature_not_s["G"]
p_a = a_s / (a_s + a_not_s)
print("Score S:", a_s)
print("Score Sc:", a_not_s)
print("P(S|A):", p_a)
b_s = p_s * (1-p_feature_s["H"]) * (1-p_feature_s["Q"]) * (1-p_feature_s["C"]) * p_feature_s["F"] * p_feature_s["J"] * (1-p_feature_s["G"])
b_not_s = p_not_s * (1-p_feature_not_s["H"]) * (1-p_feature_not_s["Q"]) * (1-p_feature_not_s["C"]) * p_feature_not_s["F"] * p_feature_not_s["J"] * (1-p_feature_not_s["G"])
p_b = b_s / (b_s + b_not_s)
print("Score S:", b_s)
print("Score Sc:", b_not_s)
print("P(S|B):", p_b)

Score S: 0.0006217168625950088
Score Sc: 0.00016177157988769763
P(S|A): 0.7935239741698317
Score S: 0.0017461300123364377
Score Sc: 0.056546133266552875
P(S|B): 0.0299547472360505


For profile A, we can see around 80% probability of having a strong sales week, with profile B having only around 3% probability\. Since profile A includes characteristics like holidays, November or Decemebr, and cold weather, its probability is much higher

We now check for the Naive assumption \(N4\)

In [24]:
p_hq_s = (train.loc[train["S"], "H"] & train.loc[train["S"], "Q"]).mean()
p_h_s = train.loc[train["S"], "H"].mean()
p_q_s = train.loc[train["S"], "Q"].mean()
print("P(H and Q | S):", p_hq_s)
print("P(H | S) * P(Q | S):", p_h_s * p_q_s)

P(H and Q | S): 0.13937282229965156
P(H | S) * P(Q | S): 0.10065983561776881


We can see how both results are different \(around 14% and 10% respectively\), meaning that H and Q are not conditionally independentent when given S\.

We now use scikit\-learn \(N5\)

In [26]:
X = ["H", "Q", "C", "F", "J", "G"]
nb = CategoricalNB(alpha=1)
nb.fit(train[X].astype(int), train["S"])
profiles = pd.DataFrame([
    [1, 1, 1, 0, 0, 1],
    [0, 0, 0, 1, 1, 0]
], columns=X)
print(nb.predict_proba(profiles))

[[0.20614944 0.79385056]
 [0.96993073 0.03006927]]


These calculations show an around 80% probabilty for profile A and around 3% probability for profile B, which match the calculations done in N3

We now evaluate on the test set

In [30]:
pred = nb.predict(test[X].astype(int))
print("Confusion Matrix:")
print(confusion_matrix(test["S"], pred))
print("Accuracy:", accuracy_score(test["S"], pred))
print("Precision:", precision_score(test["S"], pred, zero_division=0))
print("Recall:", recall_score(test["S"], pred))
never = np.zeros(len(test))
holiday = test["H"].astype(int)
print("Never Strong:")
print("Accuracy:", accuracy_score(test["S"], never))
print("Precision:", precision_score(test["S"], never, zero_division=0))
print("Recall:", recall_score(test["S"], never))
print("Holiday Rule:")
print("Accuracy:", accuracy_score(test["S"], holiday))
print("Precision:", precision_score(test["S"], holiday, zero_division=0))
print("Recall:", recall_score(test["S"], holiday))

Confusion Matrix:
[[1680    0]
 [ 255    0]]
Accuracy: 0.8682170542635659
Precision: 0.0
Recall: 0.0
Never Strong:
Accuracy: 0.8682170542635659
Precision: 0.0
Recall: 0.0
Holiday Rule:
Accuracy: 0.8454780361757106
Precision: 0.25555555555555554
Recall: 0.09019607843137255


The model achieved around 87% of accuracy, however, the results for both precision and recall show 0%\. This could happen because it had predicted all of the weeks as not strong weeks\. Since strong weeks make up a smaller proportion of the training data, the model is inherently more prone to predict a non\-strong week\. For this reason, we can see the model performned exactly the same as the never strong week baseline\. In the context of staffing decisions, the holiday rule would be more efficient\. 

Finally, we evaluate the threshold \(N7\)

In [34]:
prob = nb.predict_proba(test[X].astype(int))[:, 1]
for t in [0.1, 0.2, 0.3, 0.5]:
    pred = prob > t
    print("Threshold:", t)
    print("Precision:", round(precision_score(test["S"], pred, zero_division=0), 4))
    print("Recall:", round(recall_score(test["S"], pred), 4))

Threshold: 0.1
Precision: 0.152
Recall: 0.0745
Threshold: 0.2
Precision: 0.3
Recall: 0.0118
Threshold: 0.3
Precision: 0.0
Recall: 0.0
Threshold: 0.5
Precision: 0.0
Recall: 0.0


Lowering the threshold in our model was able to increase recall, leading to our model to be able to identify strong weeks\. However, we can see a tradeoff between recall and precision when changing the threshold\. For example, when the threshold is set at 0\.1, recall is around 7% and around 15% for recall and precision respectively\. Then, setting the threshold for 0\.2, the precision increases to 30% with the recall dropping down to around 1%\. Finally, increasing the threshold to 0\.3 and 0\.5 led to no strong weeks being predicted\. 

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=040f76bc-9900-4717-8ac0-057a74c9fbaf' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>